# V2 / V3 feature demos

Demos the experimental features layered on top of the main cohort model
(see [README](../README.md#architecture-this-repo)):

1. **Sarcasm-aware routing** (`src.sarcasm`) — shared by V2 and V3.
2. **Quantum-inspired uncertainty** (`src.quantum_uncertainty`) — shared by
   V2 and V3.
3. **Span extraction** (token-level BIO tagging) — V2's model head, decoded
   with V3's architecture-agnostic `V3.span_extraction` utilities.
4. **V3 hybrid backbone** — the BERT attention-sandwich + SSM encoder,
   through `V3.inference.Predictor`.

Sections are independent — each loads its own small model, so this
notebook is slower than the other two (multiple `bert-base-uncased` loads,
cached after the first).

In [1]:
import os
from pathlib import Path

# Notebooks live in notebooks/; every relative path in this repo
# (config.yaml, V2/config.yaml, models/, ...) assumes the process cwd
# is the repo root, same as running `python main.py ...` from the shell.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("cwd:", Path.cwd())

cwd: C:\GitHub\NLPTransformerAnalysis


## 1. Sarcasm-aware routing

`SarcasmDetector` is a rule-based feature detector (no extra model) that
scores likely sarcasm and routes the downstream prediction: trust it,
flag it with reduced confidence, or invert the sentiment.

In [2]:
from src.sarcasm import SarcasmDetector

detector = SarcasmDetector(low_threshold=0.3, high_threshold=0.7)

sarcasm_examples = [
    "Absolutely love this product! Great quality and fast shipping.",
    "Great, my package has been lost. Just great.",
    "LOVE how it broke after one day. Really 'amazing' quality.",
    "Works perfectly... not. Complete waste of money.",
]

for text in sarcasm_examples:
    result = detector.detect(text)
    print(f"{text!r}\n  {result.explanation}")

'Absolutely love this product! Great quality and fast shipping.'
  sarcasm_score=0.000 → trust
'Great, my package has been lost. Just great.'
  sarcasm_score=0.000 → trust
"LOVE how it broke after one day. Really 'amazing' quality."
  sarcasm_score=0.237 → trust (drivers: caps_ratio=0.40, quotation_sarcasm=0.40, positive_negative_clash=0.33)
'Works perfectly... not. Complete waste of money.'
  sarcasm_score=0.050 → trust (drivers: ellipsis_inversion=0.50)


## 2. Quantum-inspired uncertainty

Instead of a plain softmax, `QuantumProjection` maps the `[CLS]` embedding
to a complex state vector and builds a density matrix. Its diagonal gives
class probabilities (like softmax); its off-diagonal captures
*interference* between classes — how much the text genuinely supports
conflicting readings.

In [3]:
import torch

from src.quantum_uncertainty import QuantumProjection, classify_uncertainty

proj = QuantumProjection(input_dim=768, num_classes=3)

fake_cls = torch.randn(4, 768)  # stand-in for real BERT [CLS] embeddings
out = proj(fake_cls)

cats = classify_uncertainty(out["entropy"], out["interference"], num_classes=3)
for i in range(4):
    print(
        f"sample {i}: probs={[round(p, 3) for p in out['probs'][i].tolist()]} "
        f"entropy={out['entropy'][i]:.3f} interference={out['interference'][i]:.3f} "
        f"-> {cats['category_names'][cats['categories'][i].item()]}"
    )

sample 0: probs=[0.404, 0.521, 0.075] entropy=0.000 interference=1.272 -> ambiguous
sample 1: probs=[0.529, 0.206, 0.265] entropy=0.000 interference=1.547 -> ambiguous
sample 2: probs=[0.317, 0.093, 0.591] entropy=0.000 interference=1.185 -> ambiguous
sample 3: probs=[0.319, 0.323, 0.358] entropy=-0.000 interference=1.992 -> ambiguous


## 3. Span extraction (V2 model head)

V2's `AspectSentimentModel` adds a token-level BIO head alongside the
review-level aspect/sentiment heads, so it can point at *which words*
signal each aspect instead of just picking one aspect per review. Decoding
uses `V3.span_extraction.decode_bio_spans`, which is architecture-agnostic
(it only needs predicted BIO ids + the attention mask).

In [4]:
from transformers import AutoTokenizer

from src.data import load_config
from V2.model import build_model as build_v2_model
from V3.span_extraction import decode_bio_spans

v2_config = load_config("V2/config.yaml")
v2_model = build_v2_model(v2_config).eval()
tokenizer = AutoTokenizer.from_pretrained(v2_config["model"]["name"])

text = "Amazing camera but terrible battery life"
encoding = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=v2_config["model"]["max_seq_length"],
    return_tensors="pt",
)

with torch.no_grad():
    preds = v2_model.predict(encoding["input_ids"], encoding["attention_mask"])

spans = decode_bio_spans(
    preds["span_preds"][0],
    encoding["attention_mask"][0],
    aspect_names=v2_config["aspects"],
)
print(f"Text: {text!r}")
print("Predicted spans (untrained heads — random until the model is fine-tuned):")
for span in spans:
    tokens = tokenizer.convert_ids_to_tokens(
        encoding["input_ids"][0][span["start_token"]:span["end_token"]]
    )
    print(f"  {span['label']}: {tokens}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: 'Amazing camera but terrible battery life'
Predicted spans (untrained heads — random until the model is fine-tuned):
  B-shipping: ['battery']
  B-shipping: ['[SEP]']


## 4. V3 hybrid backbone

`V3.model_hybrid` swaps the middle BERT layers for a bidirectional SSM
block (Mamba2 on CUDA, a depthwise-conv + FFN fallback on CPU), keeping the
same aspect/sentiment/span/quantum heads as V2. `V3.inference.Predictor`
wraps it with the same interface as the main cohort's `Predictor`, plus
sarcasm/quantum/span fields.

In [5]:
from V3.inference import Predictor as V3Predictor
from V3.inference import format_prediction as v3_format_prediction

# Known issue: V3's CPU-fallback SSM block (V3/model_hybrid.py) currently
# breaks against transformers>=5.x — BertLayer's newer forward signature
# no longer matches what `_call_bert_layer` expects, so `predict_one` below
# raises RuntimeError until that compat break is fixed (tracked separately;
# not part of the V2/V3 sarcasm+quantum consolidation this notebook covers).
try:
    v3_predictor = V3Predictor.from_pretrained(config_path="V3/config_hybrid.yaml")
    result = v3_predictor.predict_one(
        "Great, my package has been lost. Just great."
    )
    print(v3_format_prediction(result, verbose=True))
except Exception as e:
    print(f"V3 hybrid backbone is currently broken against the installed transformers "
          f"version ({type(e).__name__}: {e})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


V3 hybrid backbone is currently broken against the installed transformers version (RuntimeError: Expected 2D (unbatched) or 3D (batched) input to conv1d, but got input of size: [1, 256, 256, 768])
